In [4]:
import torch
import numpy as np
import scipy.linalg as la

In [23]:
D = 16
d = 4
dt=0.01

np.random.seed(42)

M = np.random.normal(0, 1, size=(d*D*D, d*D*D)) + 1j*np.random.normal(0, 1, size=(d*D*D, d*D*D))
H = M + M.conj().T

v = np.random.normal(size=d*D*D)
v = v/la.norm(v)


In [25]:
# Using linalg
la_vec = la.expm(-1j*dt*H) @ v

In [21]:
# Using lanczos
def lanczos(v, H, dt=1, epsilon=1e-4, iter_limit=8):
    """
    Lanczos method for computing matrix exponential
    """
    dim = np.shape(v)[0]
    # build the lanczos vectors
    v0 = v / la.norm(v)
    
    vm = [v0]
    converged = False
    iter_limit = 0
    while not converged and iter_limit > len(vm):
        print('loopin')
        v = vm[-1]
        w = H @ v
        for v_i in vm:
            # subtract the projection of w onto each previous vector
            w -= np.dot(v_i.conjugate(), w) * v_i 
        norm_w = la.norm(w)
        if norm_w < epsilon:
            print(f'Norm w/epsilon: {norm_w/epsilon} ')
            converged = True
            break
        vm.append(w / norm_w)
        iter_limit += 1

    Vm = np.column_stack(vm)
    # if not Vm.shape == H_eff_mat.shape:
    #     print(f'Lanczos method saved time! Required {Vm.shape[1]} vectors')
    H_eff = Vm.conj().T @ H @ Vm
    print(f"Dimension of H representation in Lanczos space: {np.shape(H_eff)}")
    # compute the exponential exactly
    mat_exp = la.expm(-0.5*1j*dt*H_eff)
    updated_tensor = (mat_exp @ Vm.T)[:, 0]
    return updated_tensor

In [22]:
e=1e-8
iter_max=100
lanczos(v, H, dt, epsilon=e, iter_limit=iter_max)

Dimension of H representation in Lanczos space: (1, 1)


array([-0.03884866-0.00026393j])